In [1]:
import os
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

# Fetch the Groq API key
groq_api_key = os.getenv("GROQ_API_KEY")

In [2]:
from langchain_groq import ChatGroq
llm = ChatGroq(
    groq_api_key=os.getenv("GROQ_API_KEY"),
    model_name='llama-3.3-70b-versatile',
    temperature=0.1
)

In [3]:
llm.invoke("Hello, how are you today?")

AIMessage(content="Hello. I'm doing well, thank you for asking. I'm a large language model, so I don't have feelings or emotions like humans do, but I'm functioning properly and ready to assist you with any questions or tasks you may have. How can I help you today?", response_metadata={'token_usage': {'completion_tokens': 58, 'prompt_tokens': 42, 'total_tokens': 100, 'completion_time': 0.210909091, 'prompt_time': 0.00744335, 'queue_time': 0.070109279, 'total_time': 0.218352441}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_3884478861', 'finish_reason': 'stop', 'logprobs': None}, id='run-50beb274-2bb0-44a8-96a0-99379b6a7a43-0', usage_metadata={'input_tokens': 42, 'output_tokens': 58, 'total_tokens': 100})

In [4]:
llm.invoke("What is the stock price of Apple?")

AIMessage(content="I'm not able to provide real-time stock prices. However, I can suggest some ways for you to find the current stock price of Apple:\n\n1. Check online financial websites: You can visit websites like Yahoo Finance, Google Finance, or Bloomberg to get the current stock price of Apple.\n2. Use a stock market app: You can download a stock market app like Robinhood, Fidelity, or E\\*TRADE to get real-time stock prices.\n3. Visit Apple's investor relations website: Apple's investor relations website provides up-to-date information on the company's stock price, as well as other financial information.\n\nPlease note that stock prices can fluctuate rapidly and may be different from the time I provided this response. For the most accurate and up-to-date information, I recommend checking a reliable financial source.", response_metadata={'token_usage': {'completion_tokens': 166, 'prompt_tokens': 43, 'total_tokens': 209, 'completion_time': 0.603636364, 'prompt_time': 0.008511616, 

## Connecting LLMs to Financial Data

Yahoo Finance API

In [5]:
pip install yfinance

Note: you may need to restart the kernel to use updated packages.


You should consider upgrading via the 'c:\Users\KIIT\AppData\Local\Programs\Python\Python38\python.exe -m pip install --upgrade pip' command.


In [6]:
import yfinance as yf
from langchain.agents import AgentType, Tool, initialize_agent

# Initialize the LLM
llm = ChatGroq(
    groq_api_key=os.getenv("GROQ_API_KEY"),
    model_name='llama-3.3-70b-versatile',
    temperature=0.1
)

## Define the function to get the stock price
def get_stock_price(symbol):
    stock = yf.Ticker(symbol)
    today_data = stock.history(period='1d')
    return f"The current stock price of {symbol} is {today_data['Close'].iloc[-1]:.2f}"

## Create a tool for the agent to use
tools = [
    Tool(
        name="Stock Price",
        func=get_stock_price,
        description="Use this tool to get the current stock price of a given stock symbol. The input to this tool should be the stock symbol."
    )
]

## Initialize the agent
agent = initialize_agent(
    tools,
    llm,
    agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True
)

response=agent.invoke("What is the stock price of Apple?")
print(response)


C:\Users\KIIT\AppData\Local\Temp\ipykernel_32332\4282858134.py:27: LangChainDeprecationWarning: The function `initialize_agent` was deprecated in LangChain 0.1.0 and will be removed in 1.0. Use Use new agent constructor methods like create_react_agent, create_json_agent, create_structured_chat_agent, etc. instead.
  agent = initialize_agent(




> Entering new AgentExecutor chain...
To find the current stock price of Apple, I need to use the Stock Price tool. 

Action: Stock Price
Action Input: AAPL
Observation: The current stock price of AAPL is 232.80
Thought:Thought: I now know the final answer
Final Answer: The current stock price of Apple is $232.80.

> Finished chain.
{'input': 'What is the stock price of Apple?', 'output': 'The current stock price of Apple is $232.80.'}


In [7]:
from langchain_core.tools import tool, StructuredTool
from datetime import date

@tool
def company_information(ticker: str) -> dict:
    """Use this tool to retrieve company information like address, industry, sector, company officers, business summary, website,
       marketCap, current price, ebitda, total debt, total revenue, debt-to-equity, etc."""
    
    ticker_obj = yf.Ticker(ticker)
    ticker_info = ticker_obj.get_info()

    return ticker_info

@tool
def last_dividend_and_earnings_date(ticker: str) -> dict:
    """
    Use this tool to retrieve company's last dividend date and earnings release dates.
    It does not provide information about historical dividend yields.
    """
    ticker_obj = yf.Ticker(ticker)
    
    return ticker_obj.get_calendar()

@tool
def summary_of_mutual_fund_holders(ticker: str) -> dict:
    """
    Use this tool to retrieve company's top mutual fund holders. 
    It also returns their percentage of share, stock count and value of holdings.
    """
    ticker_obj = yf.Ticker(ticker)
    mf_holders = ticker_obj.get_mutualfund_holders()
    
    return mf_holders.to_dict(orient="records")

@tool
def summary_of_institutional_holders(ticker: str) -> dict:
    """
    Use this tool to retrieve company's top institutional holders. 
    It also returns their percentage of share, stock count and value of holdings.
    """
    ticker_obj = yf.Ticker(ticker)   
    inst_holders = ticker_obj.get_institutional_holders()
    
    return inst_holders.to_dict(orient="records")

@tool
def stock_grade_updrages_downgrades(ticker: str) -> dict:
    """
    Use this to retrieve grade ratings upgrades and downgrades details of particular stock.
    It'll provide name of firms along with 'To Grade' and 'From Grade' details. Grade date is also provided.
    """
    ticker_obj = yf.Ticker(ticker)
    
    curr_year = date.today().year
    
    upgrades_downgrades = ticker_obj.get_upgrades_downgrades()
    upgrades_downgrades = upgrades_downgrades.loc[upgrades_downgrades.index > f"{curr_year}-01-01"]
    upgrades_downgrades = upgrades_downgrades[upgrades_downgrades["Action"].isin(["up", "down"])]
    
    return upgrades_downgrades.to_dict(orient="records")

@tool
def stock_splits_history(ticker: str) -> dict:
    """
    Use this tool to retrieve company's historical stock splits data.
    """
    ticker_obj = yf.Ticker(ticker)
    hist_splits = ticker_obj.get_splits()
    
    return hist_splits.to_dict()

@tool
def stock_news(ticker: str) -> dict:
    """
    Use this to retrieve latest news articles discussing particular stock ticker.
    """
    ticker_obj = yf.Ticker(ticker)
    
    return ticker_obj.get_news()

In [8]:
pip install phi

Note: you may need to restart the kernel to use updated packages.


You should consider upgrading via the 'c:\Users\KIIT\AppData\Local\Programs\Python\Python38\python.exe -m pip install --upgrade pip' command.


In [19]:
# from phi import Agent
import phi
from phi.agent import Agent, create_tool_calling_agent 

ModuleNotFoundError: No module named 'phi.agent'

In [ ]:
from langchain.agents import AgentExecutor, initialize_agent, AgentType

# Assuming you've already defined your tools and LLM:
tools = [
    company_information,
    last_dividend_and_earnings_date,
    stock_splits_history,
    summary_of_mutual_fund_holders,
    summary_of_institutional_holders, 
    stock_grade_updrages_downgrades,
    stock_news,
    get_stock_price
]

# Create the financial agent using initialize_agent
finance_agent = initialize_agent(
    tools=tools,
    llm=llm,
    agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True
)

# Execute the agent
query = "Give me Apple's company information and its current stock price."
response = finance_agent.run(query)

print(response)


AttributeError: 'function' object has no attribute 'is_single_input'

: 